# Treinamento do Modelo

## Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
# from sklearn.preprocessing import LabelEncoder
from scipy import stats

# Funções

In [ ]:
# root_dir = "/content/drive/MyDrive/pcaps"
# input_dataset = f"{root_dir}/features-dash-sem-wave-v4.csv"
# output_tree = f"{root_dir}/tree.txt"

In [ ]:
def removeAtributoscomMesmoValor(dataset):

    dataset.drop(columns=dataset.columns[dataset.nunique()==1], inplace=True)

    return dataset

In [ ]:
def removeOutliers(dataset):

    # detect and remove outliers
    z_scores = stats.zscore(dataset)
    abs_z_scores = np.abs(z_scores)
    filtered_entries = (abs_z_scores < 5).all(axis=1)
    dataset = dataset[filtered_entries]
    # reset index
    #dataset = dataset.reset_index(drop=True)

    return dataset

In [ ]:
def removeDadosFaltantes(dataset):

    dataset=dataset.dropna()
    return dataset

In [ ]:
def normalization(X):

    scaler = sk.preprocessing.StandardScaler().fit(X)
    X = scaler.transform(X)
    return X

In [ ]:
def test_model(model, X, y, plot_matrix=False):
  y_pred = model.predict(X)

  accuracy = accuracy_score(y, y_pred)
  precision = precision_score(y, y_pred, average="weighted")
  recall = recall_score(y, y_pred, average="weighted")
  f1 = f1_score(y, y_pred, average="weighted")

  cm = confusion_matrix(y, y_pred)

  # Display metrics
  print("\n" + "="*50)
  print("MODEL PERFORMANCE METRICS")
  print("="*50)
  print(f"Accuracy:  {accuracy:.4f}")
  print(f"Precision: {precision:.4f}")
  print(f"Recall:    {recall:.4f}")
  print(f"F1-Score:  {f1:.4f}")

  print("\n" + "="*50)
  print("DETAILED CLASSIFICATION REPORT")
  print("="*50)
  print(classification_report(y, y_pred))

  print("\n" + "="*50)
  print("CONFUSION MATRIX")
  print("="*50)
  print(cm)

  # Feature importance
  feature_importance = pd.DataFrame({
      'feature': X.columns,
      'importance': model.feature_importances_
  }).sort_values('importance', ascending=False)

  print("\n" + "="*50)
  print("FEATURE IMPORTANCE")
  print("="*50)
  print(feature_importance)

  if plot_matrix:
    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=np.unique(y),
                yticklabels=np.unique(y))
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/confusion_matrix.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def import_csv(file, show_info=False):
  data = pd.read_csv(file)

  # data.dropna(subset=["flags"], inplace=True)

  # data = removeAtributoscomMesmoValor(data)
  data = removeDadosFaltantes(data)
  # data["IPI"] = removeOutliers(data["IPI"])

  if show_info:
    print("Info:")
    data.info()
    print("\nDescrição estatística:")
    print(data.describe())
    print("\nValores nulos:")
    print(data.isnull().sum())
    print("\nDistribuição da target:")
    print(data['classification'].value_counts())
    # print(data["Flags"].value_counts())

  return data

## Importação dos dados

In [ ]:
data = import_csv(input_dataset, True)

In [ ]:
# data2 = import_csv(input_dataset2, True)

# Treino com melhor par de features

In [ ]:
# from itertools import combinations

# all_features = [
#     # 'sport',
#     # 'dport',
#     'tos',
#     # 'length',
#     # 'id',
#     # 'ttl',
#     'frame_size',
#     # 'chksum',
#     # 'seq',
#     # 'ack',
#     'window',
#     'ipi'
# ]

# target = "classification"

# X = data[all_features]
# y = data[target]
# X2 = data2[all_features]
# y2 = data2[target]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.3, random_state=64,# stratify=y
# )

# max_depth=10

# current_features = []

# model_pair = DecisionTreeClassifier(random_state=64, max_depth=max_depth)

# for i in range(5):
#   feature_pairs = list(combinations(all_features, 2))
#   print(f"Generated {len(feature_pairs)} unique feature pairs.")

#   best_accuracy = 0
#   best_validation = 0
#   best_feature_pair = None

#   for pair in feature_pairs:
#       X_train_pair = X_train[current_features + list(pair)]
#       X_test_pair = X_test[current_features + list(pair)]
#       X2_pair = X2[current_features + list(pair)]

#       model_pair.fit(X_train_pair, y_train)

#       y_pred_pair = model_pair.predict(X_test_pair)
#       y2_pred_pair = model_pair.predict(X2_pair)

#       accuracy_pair = accuracy_score(y_test, y_pred_pair)
#       accuracy2_pair = accuracy_score(y2, y2_pred_pair)

#       # accuracy_pair = recall_score(y_test, y_pred_pair, average=None)[1]
#       # accuracy2_pair = recall_score(y2, y2_pred_pair, average=None)[1]

#       if accuracy_pair > best_accuracy:
#           best_accuracy = accuracy_pair
#           best_feature_pair = pair
#           best_validation = accuracy2_pair




#   # Add the best feature pair to the current features list
#   current_features.append(best_feature_pair[0])
#   current_features.append(best_feature_pair[1])

#   # Remove best feature pair from the all_features list
#   all_features.remove(best_feature_pair[0])
#   all_features.remove(best_feature_pair[1])

#   print(f"Current execution: {i+1}")
#   print(f"Current features: {current_features}")
#   print(f"Best feature pair: {best_feature_pair}")
#   print(f"Highest accuracy achieved: {best_accuracy:.4f}")
#   print(f"Highest accuracy achieved on validation: {best_validation:.4f}")
#   print(f"Remaining features: {all_features}")
#   print("="*50)

In [ ]:
# features = [
# # "sport"         ,
# # "dport"         ,
# "tos"           ,
# # "length"        ,
# # "id"            ,
# # "ttl"           ,
# # "frame_size"    ,
# # "chksum"        ,
# # "seq"           ,
# # "ack"           ,
# "window"        ,
# # "ipi"           ,
# # "dataofs"       ,
# # "urgptr"        ,
# # "reserved"      ,
# # "tcp_chk"       ,
# # "proto"         ,
# # "version"       ,
# # "frag"          ,
# # "ihl"           ,
# #  "classification",
# ]

max_depth=10

class_names = [
    "nao_video",
    "video",
]

target = "classification"

X = data[features]
y = data[target]
# X2 = data2[features]
# y2 = data2[target]

# Treino do modelo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,# stratify=y
)

model = DecisionTreeClassifier(random_state=42, max_depth=max_depth)
model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=10, random_state=42)

## Teste do modelo

### Com base de treinos

In [ ]:
test_model(model, X_train, y_train, True)

### Com base de Teste

In [ ]:
test_model(model, X_test, y_test, True)

# Teste com dataset diferente

In [ ]:
# test_model(model, X2, y2, True)

# Exportar Modelo

In [ ]:
from sklearn.tree import _tree
from datetime import datetime
import pickle

def exportar_regras_modelo(
    modelo: DecisionTreeClassifier,
    features: list[str],
    # class_names: list[str],
    ):

  def _add_value(table, key, value):
    # value = int(new_value)
    if key not in table:
      table[key] = []
      table[key].append(value)
      return

    if value not in table[key]:
      table[key].append(value)

  res = {}

  regras = []

  def _recursive(node, conditions):
    if modelo.tree_.children_left[node] == _tree.TREE_LEAF:
      regras.append(f"when {' and '.join(conditions)} then {modelo.tree_.value[node].argmax()}")
      return

    feature = features[modelo.tree_.feature[node]]
    threshold = modelo.tree_.threshold[node]

    if feature == "ipi":
      threshold = int(threshold)
    else:
      threshold = int(threshold)

    _add_value(res, feature, threshold)

    left_conditions = conditions + [f"{feature}<={threshold}"]
    _recursive(modelo.tree_.children_left[node], left_conditions)

    right_conditions = conditions + [f"{feature}>{threshold}"]
    _recursive(modelo.tree_.children_right[node], right_conditions)

  _recursive(0, [])

  res["regras"] = regras

  for feature in features:
    try:
      res[feature] = sorted(res[feature])
    except:
      res[feature] = []

  return res

res = exportar_regras_modelo(model, features)

with open(output_tree, "w") as tree:
  for feature in features:
    tree.write(f"{feature} = {res[feature]};\n")
  for regra in res["regras"]:
    tree.write(f"{regra};\n")

for feature in features:
  print(f"{feature} = {res[feature]};")

for regra in res["regras"]:
  print(regra)

with open(f"{output_tree.replace('txt', 'pkl')}", "wb") as f:
  pickle.dump(model, f)

In [ ]:
# tree_rules = export_text(model, feature_names=features, class_names=class_names)

# print(tree_rules)

In [ ]:
# from sklearn.tree import _tree

# print(model.tree_.max_depth)
# print(model.tree_.threshold)
# print(_tree.TREE_UNDEFINED)
# print(_tree.TREE_LEAF)
# print(model.tree_.children_left)
# print(model.tree_.children_right)
# print(model.tree_.node_count)
# print(model.tree_.feature)

In [ ]:
import pickle

with open(f"{output_tree.replace('txt', 'pkl')}", "wb") as f:
  pickle.dump(model, f)